# Kaggle runner — E3: feature-signal sweep

Clones the `dphgnn_gf` branch, sets up the env (same pattern as `lifting_confounding_study/kaggle_run_lifting_ablation.ipynb`), then runs the E3 experiment: `run_e3.py` (training) followed by `feature_signal.ipynb` (analysis + figure).

See `2026_tdl_challenge/extra_analysis_oversmooth_operators/feature_signal/README.md` for the experiment design (5 `center_variance` points x 2 models, H1/H2).

**Run the cells top to bottom. Do not skip the preflight/smoke-test cell** — `run_e3.py` runs `preflight_check()` automatically, which verifies the `center_variance` Hydra override path actually takes effect (acceptance tests A and B), checks each sweep point gets its own preprocessing cache directory, and attempts the empirical `feature_signal` diagnostic. A silent override failure here would produce a flat curve that looks like a real null result — this is the single most dangerous failure mode in this experiment.

In [ ]:
import os, shutil, subprocess
os.chdir("/kaggle/working")
print("cwd =", os.getcwd())

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Pas de GPU attaché — vérifie Accelerator dans le panneau de droite.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## 1. Clone the branch and set up the environment

In [ ]:
%%bash
cd /kaggle/working          # <<< INDISPENSABLE : recale le sous-shell
set -eo pipefail

DEST=/kaggle/working/topobench
BRANCH=e2_e3_experiments
REPO=https://github.com/yeli-falk/topobench.git
CUDA_VARIANT=cu118

export UV_CACHE_DIR=/tmp/uv-cache
export UV_LINK_MODE=copy

if [ -d "$DEST/.git" ]; then
  git -C "$DEST" fetch origin "$BRANCH"
  git -C "$DEST" checkout "$BRANCH"
  git -C "$DEST" reset --hard "origin/$BRANCH"
else
  rm -rf "$DEST"
  git clone --branch "$BRANCH" "$REPO" "$DEST"
fi

cd "$DEST"

pip install -q -U uv
export PATH="$HOME/.local/bin:$PATH"
uv --version

source uv_env_setup.sh "$CUDA_VARIANT"

# psutil: hard dependency of the orchestrator (RAM diagnostics between
# jobs), not in pyproject.toml. statsmodels is optional (nicer Phase-2
# stats table; the notebook falls back to a manual computation if absent).
uv pip install -q ipykernel nbconvert jupyter-client psutil statsmodels
python -m ipykernel install --user --name=topobench --display-name "Python 3.11 (topobench)"

python - <<'PY'
import torch
print(f"Torch {torch.__version__} | CUDA dispo: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'}")
PY

# uv_env_setup.sh rewrites pyproject.toml's PyG find-links/index in
# place for the chosen CUDA variant -- expected to show as modified,
# do not commit it back.
git status --porcelain pyproject.toml


## 2. Preflight + smoke test on the real Kaggle GPU (recommended, ~5-10 min)

Runs `preflight_check()` (acceptance tests A/B on `center_variance`, cache-directory distinctness, the empirical `feature_signal` diagnostic — none of this touches the GPU) followed by a smoke test: tiny synthetic data + 2 epochs across all 5 points x 2 models. Nothing is written to `e3_feature_signal_results.json` by a smoke test. **If this cell errors, stop** — do not proceed to Phase 1 below.

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/feature_signal
../../../.venv/bin/python -u run_e3.py --smoke-test


## 3. Phase 1 — the real sweep (10 runs, seed 42)

5 `center_variance` points (`fs_00`..`fs_04`) x 2 models (`hypergraph/dphgnn`, `graph/gcn`) x seed 42. Each `center_variance` value regenerates a whole GraphUniverse universe (not a resample), so dataset generation runs 5 times — see README.md for the timing projection. Resumable: safe to stop and re-run this cell, already-completed `(point, model, seed)` triples are skipped.

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/feature_signal
../../../.venv/bin/python -u run_e3.py


## 4. Phase 2 (optional — only run once Phase 1 above is fully green)

Adds seeds 43 and 44 (30 runs total). Enables the bootstrap 95% CI on `Δ = DPHGNN - GCN` at each point (see README.md, "Statistics"). Skip this cell if you're short on GPU budget — Phase 1 alone is still reportable, just descriptive (n=1 seed, no crossover point quoted numerically).

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/feature_signal
../../../.venv/bin/python -u run_e3.py --phase2


## 5. Analysis — populate the figure and the finding

Only reads `e3_feature_signal_results.json` and `feature_signal_empirical.json` and plots — never trains. Writes `figures/fig5_feature_signal_crossover.png` in place.

In [ ]:
%%bash
cd /kaggle/working/topobench/2026_tdl_challenge/extra_analysis_oversmooth_operators/feature_signal
../../../.venv/bin/python -m jupyter nbconvert \
    --to notebook --execute --inplace \
    --ExecutePreprocessor.kernel_name=topobench \
    --ExecutePreprocessor.timeout=600 \
    feature_signal.ipynb


## 6. Sanity check the outputs

In [ ]:
import json
from pathlib import Path

exp_dir = Path(
    "/kaggle/working/topobench/2026_tdl_challenge/"
    "extra_analysis_oversmooth_operators/feature_signal"
)

results_path = exp_dir / "e3_feature_signal_results.json"
if results_path.exists():
    records = json.loads(results_path.read_text())
    ok = sum(1 for r in records if r.get("status") == "ok")
    failed = [r for r in records if r.get("status") != "ok"]
    print(f"{results_path}: {len(records)} record(s), {ok} ok, {len(failed)} failed")
    for r in failed:
        print(f"  FAILED {r.get('point')}/{r.get('model')}/s{r.get('seed')}: {r.get('error')}")
else:
    print(f"{results_path} does not exist yet — run Phase 1 (cell above) first.")

figures_dir = exp_dir / "figures"
figs = sorted(figures_dir.glob("*.png")) if figures_dir.exists() else []
print(f"\nFigures in {figures_dir}:")
for f in figs:
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not figs:
    print("  (none yet — run the analysis cell above)")
